#### This project is aimed to generate finetuning alloy machine learning for field for alloy systems. We will start from the Li-Ga system first.


In [ ]:
import os
os.mkdir("LiGa")

In [ ]:
!ls

In [ ]:
!conda activate dp_qqz

In [3]:
#First of all, we want to grab all "reasonable" Li-Ga alloy systems
import os
from mp_api.client import MPRester

# 1. Setup Output Directory
output_dir = "LiGa/structures"
os.makedirs(output_dir, exist_ok=True)

# 2. Query Materials Project
# Replace 'YOUR_API_KEY' with your actual key if not using environment variables
# (I have removed your key from this snippet for security)
with MPRester("bTeurW7C1cBsIosJYTS5jW0LY7XJ8CCt") as mpr:
    print("Searching for pure Li (ground state)...")
    docs_Li = mpr.materials.summary.search(chemsys="Li", energy_above_hull=(0, 0))
    
    print("Searching for pure Ga (ground state)...")
    docs_Ga = mpr.materials.summary.search(chemsys="Ga", energy_above_hull=(0, 0))
    
    print("Searching for Li-Ga alloys (stable & metastable < 20 meV)...")
    docs_comp = mpr.materials.summary.search(chemsys="Li-Ga", energy_above_hull=(0, 0.02))

    # Combine all found documents
    all_docs = docs_Li + docs_Ga + docs_comp
    print(f"Found {len(all_docs)} total structures.")

    # 3. Save Structures
    for doc in all_docs:
        # Extract the Structure object (Pymatgen object)
        structure = doc.structure
        material_id = doc.material_id
        formula = doc.formula_pretty

        # Define filename (e.g., "LiGa_mp-1234.cif")
        # CIF is robust for storage; you can also use .vasp or .xyz
        filename = f"{formula}_{material_id}.cif"
        file_path = os.path.join(output_dir, filename)
        
        # Save file
        structure.to(filename=file_path)
        print(f"Saved: {filename}")

print(f"\nAll structures stored in '{output_dir}'")

/dssg/home/acct-matxzl/matxzl/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'NotRequired' from 'typing' (/dssg/home/acct-matxzl/matxzl/.conda/envs/dp_qqz/lib/python3.10/typing.py)

In [4]:
import os
import warnings
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet
import math
# 1. Setup Directories
# input_dir = "LiGa/structures"  <-- 原代码
input_dir = "Na_Na3SbS4_Data/structures" # <-- 修改后

# base_output_dir = "LiGa/AIMD_runs" <-- 原代码
base_output_dir = "Na_Na3SbS4_Data/AIMD_runs" # <-- 修改后  # Separate folder to keep things clean
os.makedirs(base_output_dir, exist_ok=True)

# 2. Define AIMD Settings (Overrides standard RelaxSet)
# These settings ensure we sample the potential energy surface, not just find the minimum.
aimd_settings_high_T = {
    "IBRION": 0,          # MD 模式
    "NSW": 2000,          # 步数 (2000步 * 2fs = 4ps，对于训练集生成通常足够)
    "POTIM": 2.0,         # 时间步长 (fs)
    "TEBEG": 800,         # 起始温度 800K (远高于 Na 熔点，确保采样液态构型)
    "TEEND": 800,         # 结束温度
    "ISYM": 0,            # 关闭对称性 (必须)
    "SMASS": 0,           # Nose-Hoover 热浴
    "LREAL": "Auto",      # 投影算符自动选择 (加速计算)
    "ALGO": "Fast",       # 电子步算法
    "PREC": "Normal",     # 精度
    "ISIF": 2,            # 固定体积 (训练数据建议固定体积，防止高温下盒子塌陷或无限膨胀)
    "KSPACING": 0.4,      # 使用 KSPACING 而不是 KPOINTS，0.4 对应 Gamma 点附近，大体系够用了
    "ISMEAR": 0,          # Gaussian Smearing
    "SIGMA": 0.05,        # 展宽
    "ENCUT": 520,         # 截断能 (Materials Project 兼容)
    "LWAVE": False,       # 不保存波函数 (省空间)
    "LCHARG": False,      # 不保存电荷密度
    "NELM": 60,           # 电子步最大迭代次数 (高温下难收敛，稍微调大)
}

# 3. Processing Loop
print(f"Reading structures from {input_dir}...")

for filename in os.listdir(input_dir):
    if filename.endswith(".cif") or filename.endswith(".vasp"):
        file_path = os.path.join(input_dir, filename)
        struct_name = os.path.splitext(filename)[0]
        
        try:
            # Load Structure
            structure = Structure.from_file(file_path)
            
            # --- CRITICAL FOR MLIP: SUPERCELL GENERATION ---
            # MLIPs have a cutoff radius (usually 4-6 Angstroms).
            # If the cell is smaller than 2x Cutoff, atoms see themselves.
            # We enforce a minimum image distance > 10 Angstroms.
            # --- FIX: Manual Supercell Calculation ---
            # We want the minimum dimension to be at least 10.0 Angstroms
            # to avoid self-interaction artifacts in MACE.
            min_length = 10.0
            
            # Get current lattice lengths (a, b, c)
            lengths = structure.lattice.abc
            
            # Calculate scaling factors: ceil(10.0 / length)
            # Example: if length is 3.0, scaling is ceil(3.33) = 4
            scaling_matrix = [max(1, int(math.ceil(min_length / l))) for l in lengths]
            
            # Apply the supercell
            structure.make_supercell(scaling_matrix)
            
            # Create Output Subdirectory
            task_dir = os.path.join(base_output_dir, struct_name)
            os.makedirs(task_dir, exist_ok=True)
            
            # Generate VASP Inputs
            # We use MPRelaxSet as a base because it handles POTCARs/KPOINTS well,
            # but we strictly override the INCAR for MD.
            vis = MPRelaxSet(
                structure, 
                user_incar_settings=aimd_settings,
                user_kpoints_settings={"reciprocal_density": 50},
                user_potcar_functional="PBE"
            )
            
            # Write input files
            vis.write_input(task_dir)
            print(f"Generated inputs for: {struct_name} (Supercell: {structure.num_sites} atoms)")
            
        except Exception as e:
            print(f"Skipped {filename}: {e}")

print(f"\nDone. VASP inputs are ready in '{base_output_dir}'.")

Reading structures from LiGa/structures...
Generated inputs for: Li2Ga7_mp-1222769 (Supercell: 144 atoms)
Generated inputs for: LiGa3_mp-863257 (Supercell: 108 atoms)
Generated inputs for: Li_mp-1018134 (Supercell: 96 atoms)
Generated inputs for: Ga_mp-142 (Supercell: 108 atoms)
Generated inputs for: LiGa_mp-1307 (Supercell: 108 atoms)
Generated inputs for: Li2Ga_mp-29210 (Supercell: 108 atoms)
Generated inputs for: Li3Ga14_mp-1222489 (Supercell: 136 atoms)
Generated inputs for: Li5Ga4_mp-1205930 (Supercell: 162 atoms)
Generated inputs for: Li3Ga2_mp-9568 (Supercell: 40 atoms)

Done. VASP inputs are ready in 'LiGa/AIMD_runs'.


### Submit tasks

In [ ]:
import os
import glob
from dpdispatcher import Machine, Resources, Task, Submission

# 1. Define the Machine (Slurm + LazyLocal)
# LazyLocal means we run directly on the cluster without SSH-ing to another machine.
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

# 2. Define Resources
# We map your previous custom_flags and settings here.
# Note: Total cores = number_node * cpu_per_node = 2 * 64 = 128
resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g",  # This sets the partition
    group_size=1,          # 1 task per job (no bundling)
    module_list=["oneapi/2021.4.0"],
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=sunzhetao1997@sjtu.edu.cn",
        # Standard VASP parallelization flags can be added here if needed
        "#SBATCH --ntasks=64" 
    ]
)

# 3. Define the Command
# Adjust 'vasp_std' to your specific executable (e.g., vasp_gam, vasp_std)
command = "mpirun -np 64 vasp_std"

# 4. Collect Tasks
task_list = []
work_base = "LiGa/AIMD_runs"
search_pattern = os.path.join(work_base, "*")

print(f"Scanning {work_base} for tasks...")

for folder_path in glob.glob(search_pattern):
    if os.path.isdir(folder_path):
        folder_name = os.path.basename(folder_path)
        
        # Create a Task for each folder
        # forward_files=[] means we use the files ALREADY in the directory
        # backward_files defines what we expect to verify/download (good practice)
        task = Task(
            command=command,
            task_work_path=folder_name,
            forward_files=[], 
            backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR']
        )
        task_list.append(task)

print(f"Found {len(task_list)} tasks.")

# 5. Create and Run Submission
if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )

    print("Submitting tasks to Slurm...")
    submission.run_submission()
    print("Submission finished.")
else:
    print("No subdirectories found in LiGa/AIMD_runs/")

Scanning LiGa/AIMD_runs for tasks...
Found 9 tasks.
Submitting tasks to Slurm...
2025-11-18 14:12:38,148 - INFO : info:check_all_finished: False
2025-11-18 14:12:38,194 - INFO : job: 52eaba5f52156aa3327983134015e882253cfa09 submit; job_id is 50219079
2025-11-18 14:12:38,233 - INFO : job: dddfa8825f334888857b6129f5258f0b8d8e69e8 submit; job_id is 50219080
2025-11-18 14:12:38,260 - INFO : job: 51e2d6381787668f824db74fb73ed3a472c1885e submit; job_id is 50219081
2025-11-18 14:12:38,289 - INFO : job: 3b688253d282a6c7fcb38daf866652f42cd9dcdd submit; job_id is 50219082
2025-11-18 14:12:38,315 - INFO : job: 168af98c6af2d7bb66c23af903348df3d253b107 submit; job_id is 50219083
2025-11-18 14:12:38,341 - INFO : job: c528c843f07fae8d460ec92cfbc3d313e4abcb31 submit; job_id is 50219084
2025-11-18 14:12:38,367 - INFO : job: 7ddb1312a0481bdac7554b83c84a251c7d23afc3 submit; job_id is 50219085
2025-11-18 14:12:38,399 - INFO : job: a29e15343796ec805734e74fceb63e79ea1f7488 submit; job_id is 50219086
2025-11

In [ ]:
import os
import glob
import numpy as np
from ase.io import read, write

In [4]:
import os
import glob
import numpy as np
from ase.io import read, write

# 1. Settings
root_dir = "LiGa/AIMD_runs"
output_train = "test4/LiGa_train.xyz"
output_valid = "test4/LiGa_val.xyz"
validation_ratio = 0.1   # 10% for validation
shuffle_seed = 42        # Fixed seed for reproducibility

# --- NEW PARAMETER ---
# Set to an integer (e.g., 1000) to limit dataset size. 
# Set to None to use all available frames.
target_total_frames = 4000 

# 2. Collect all trajectories
print(f"Scanning {root_dir} for vasprun.xml files...")
search_pattern = os.path.join(root_dir, "*", "vasprun.xml")
vasp_files = glob.glob(search_pattern)

if not vasp_files:
    print("No vasprun.xml files found! Check your directory or if VASP jobs finished.")
    exit()

all_atoms = []

for i, vasp_file in enumerate(vasp_files):
    try:
        # Read trajectory
        traj = read(vasp_file, index=':')
        
        # Optional: Skip equilibration (uncomment if needed)
        # traj = traj[25:] 
        
        all_atoms.extend(traj)
        print(f"[{i+1}/{len(vasp_files)}] Loaded {len(traj)} frames from {os.path.basename(os.path.dirname(vasp_file))}")
        
    except Exception as e:
        print(f"Error reading {vasp_file}: {e}")

total_found = len(all_atoms)
print(f"Total configurations collected: {total_found}")

# 3. Shuffle and Subsample
# We shuffle FIRST to ensure we pick a random selection from all trajectories
np.random.seed(shuffle_seed)
np.random.shuffle(all_atoms)

# --- NEW LOGIC: Apply Frame Limit ---
if target_total_frames is not None and total_found > target_total_frames:
    print(f"Downsampling dataset from {total_found} to {target_total_frames} frames...")
    all_atoms = all_atoms[:target_total_frames]
    final_count = target_total_frames
else:
    print(f"Using all {total_found} frames (Target was {target_total_frames}).")
    final_count = total_found

# 4. Split into Train/Val
n_valid = int(final_count * validation_ratio)
n_train = final_count - n_valid

train_set = all_atoms[:n_train]
valid_set = all_atoms[n_train:]

print(f"Final Dataset Split: {n_train} Training, {n_valid} Validation")

# 5. Save to Extended XYZ
print(f"Writing {output_train}...")
write(output_train, train_set, format='extxyz')

print(f"Writing {output_valid}...")
write(output_valid, valid_set, format='extxyz')

print("Done! Dataset generation complete.")

Scanning LiGa/AIMD_runs for vasprun.xml files...
[1/9] Loaded 5000 frames from LiGa3_mp-863257
[2/9] Loaded 5000 frames from Li2Ga_mp-29210
[3/9] Loaded 5000 frames from Li2Ga7_mp-1222769
[4/9] Loaded 5000 frames from Ga_mp-142
[5/9] Loaded 5000 frames from Li3Ga14_mp-1222489
[6/9] Loaded 5000 frames from Li3Ga2_mp-9568
[7/9] Loaded 5000 frames from LiGa_mp-1307
[8/9] Loaded 5000 frames from Li5Ga4_mp-1205930
[9/9] Loaded 5000 frames from Li_mp-1018134
Total configurations collected: 45000
Downsampling dataset from 45000 to 4000 frames...
Final Dataset Split: 3600 Training, 400 Validation
Writing test4/LiGa_train.xyz...
Writing test4/LiGa_val.xyz...
Done! Dataset generation complete.


In [ ]:
import os
from dpdispatcher import Machine, Resources, Task, Submission

# --- USER CONFIGURATION ---
# 1. Files
path="./test4"
train_file = "LiGa_train.xyz"
valid_file = "LiGa_val.xyz"
foundation_model = "./mace-mpa-0-medium.model"
job_name = "LiGa_MACE_FT"

# 2. Isolated Atom Energies (MUST UPDATE THESE!)
# Format: {AtomicNumber: Energy_eV} -> Li=3, Ga=31
# Example: "{3: -1.90, 31: -2.85}"
E0s = "{3: -1.902, 31: -2.845}" 

# 3. Hyperparameters (Based on Tutorial Table III)
# "medium" model (L=1) is standard for MPA-0.
command_list = [
    "source activate mace2 &&", 
    "export MPLBACKEND=Agg &&",
    f"cd {path}&&",
    f'''
    python3 -m mace.cli.run_train \
    --name='LiGa_MACE_FT' \
    --foundation_model='{foundation_model}' \
    --train_file='{train_file}' \
    --valid_file='{valid_file}' \
    --E0s='{E0s}' \
    --energy_key='energy' \
    --forces_key='forces' \
    --stress_key='stress' \
    --energy_weight=1.0 \
    --forces_weight=10.0 \
    --loss='universal' \
    --lr=0.0005 \
    --batch_size=10 \
    --max_num_epochs=100 \
    --ema \
    --ema_decay=0.99 \
    --device=cuda \
    --default_dtype='float64' \
    --checkpoints_dir='checkpoints' \
    --results_dir='results' \
    --model_dir='./'
    '''
]


# Join command list into a single string
command = " ".join(command_list)

# --- DPDISPATCHER SETUP ---

machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

resources = Resources(
    number_node=1,
    cpu_per_node=16,          # Needs fewer CPUs than VASP
    gpu_per_node=1,           # REQUEST GPU
    queue_name="a100",     # Replace with your GPU partition name!
    group_size=1,
    module_list=[],           # Add 'cuda/11.7' or similar if needed outside conda
    custom_flags=[
        f"#SBATCH --job-name={job_name}",
        "#SBATCH --partition=a100", # UPDATE THIS
        "#SBATCH --gres=gpu:1",
        # "#SBATCH --mem=32G",
        # "#SBATCH --time=0:00:00",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=sunzhetao1997@sjtu.edu.cn"
    ]
)

# Define the Task
task = Task(
    command=command,
    task_work_path="./",
)

# Submit
submission = Submission(
    work_base=os.getcwd(),
    machine=machine,
    resources=resources,
    task_list=[task]
)

print(f"Submitting {job_name} to Slurm...")
submission.run_submission()
print("Done.")

Submitting LiGa_MACE_FT to Slurm...
2025-11-20 16:26:46,335 - INFO : info:check_all_finished: False
2025-11-20 16:26:46,392 - INFO : job: 53831987b4f9b0783f04c1a6165f2a82fb31fa4b submit; job_id is 50261599


In [2]:
# run data_process.py to process data
import os
from dpdispatcher import Machine, Resources, Task, Submission
job_name='data_process'
# Join command list into a single string
command = "source activate mace2 && python data_porcess.py"
# --- DPDISPATCHER SETUP ---

machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

resources = Resources(
    number_node=1,
    cpu_per_node=16,          # Needs fewer CPUs than VASP
    gpu_per_node=1,           # REQUEST GPU
    queue_name="a100",     # Replace with your GPU partition name!
    group_size=1,
    module_list=[],           # Add 'cuda/11.7' or similar if needed outside conda
    custom_flags=[
        f"#SBATCH --job-name={job_name}",
        "#SBATCH --partition=a100", # UPDATE THIS
        "#SBATCH --gres=gpu:1",
        # "#SBATCH --mem=32G",
        # "#SBATCH --time=0:00:00",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=sunzhetao1997@sjtu.edu.cn"
    ]
)

# Define the Task
task = Task(
    command=command,
    task_work_path="./",
)

# Submit
submission = Submission(
    work_base=os.getcwd(),
    machine=machine,
    resources=resources,
    task_list=[task]
)

print(f"Submitting {job_name} to Slurm...")
submission.run_submission()
print("Done.")

Submitting data_process to Slurm...
2025-11-21 00:22:13,510 - INFO : info:check_all_finished: False
2025-11-21 00:22:13,638 - INFO : job: 47977e548528bb2ac0bc778966e455f8b969bfe7 submit; job_id is 50267162
2025-11-21 00:36:12,162 - ERROR : 
Traceback (most recent call last):
  File "/dssg/home/acct-umjbsh/umjbsh-sunzht/.conda/envs/mace/lib/python3.11/site-packages/dpdispatcher/submission.py", line 248, in run_submission
    time.sleep(check_interval)
KeyboardInterrupt
2025-11-21 00:36:12,163 - INFO : submission exit: 62fbb147877f5b5ad19b0fdb7e5e5697d0cc493b
2025-11-21 00:36:12,164 - INFO : at /dssg/home/acct-umjbsh/umjbsh-sunzht/MACE_alloy
2025-11-21 00:36:12,164 - INFO : Submission information is saved in /dssg/home/acct-umjbsh/umjbsh-sunzht/.dpdispatcher/submission/62fbb147877f5b5ad19b0fdb7e5e5697d0cc493b.json.


KeyboardInterrupt: 

### Now lets do md simulation

In [21]:
import os
import math
import random
import warnings
import numpy as np
from pymatgen.core import Structure, Composition
from pymatgen.io.lammps.data import LammpsData

# Suppress warnings
warnings.filterwarnings("ignore")

# --- CONFIGURATION ---
input_dir = "LiGa/structures"
output_base = "LiGa/MD_simulation"
min_atoms = 5000  # Target system size
model_filename = "/dssg/home/acct-umjbsh/umjbsh-sunzht/MACE_alloy/LiGa/MD_simulation/LiGa_MACE_FT.model-lammps.pt"

def get_random_alloy(disordered_struct):
    """ Converts disordered structure to ordered supercell matching stoichiometry. """
    if disordered_struct.is_ordered:
        return disordered_struct
    
    total_comp = Composition()
    for site in disordered_struct:
        for species, occupancy in site.species.items():
            total_comp += Composition({species: occupancy})
            
    target_counts = {}
    for species, amount in total_comp.items():
        target_counts[species] = int(round(amount))
    
    diff = len(disordered_struct) - sum(target_counts.values())
    if diff != 0:
        most_abundant = max(target_counts, key=target_counts.get)
        target_counts[most_abundant] += diff
        
    atom_pool = []
    for species, count in target_counts.items():
        atom_pool.extend([species] * count)
    random.shuffle(atom_pool)
    
    ordered = disordered_struct.copy()
    for i, species in enumerate(atom_pool):
        ordered.replace(i, species)
    return ordered

def write_lammps_input(folder, struct_name, model_file, atom_types):
    """ 
    Writes the in.lammps file with MSD calculation for Lithium.
    atom_types: Dict mapping Element Symbol -> LAMMPS Type ID (e.g., {'Li': 2, 'Ga': 1})
    """
    
    # Determine which type ID corresponds to Lithium
    li_type_id = atom_types.get("Li", 1) # Default to 1 if not found
    
    with open(os.path.join(folder, "in.lammps"), "w") as f:
        f.write(f"""# MACE-MD for {struct_name} with Li MSD
units           metal
boundary        p p p
atom_style      atomic
atom_modify     map yes
newton          on

# Read Structure
read_data       data.lammps

# Interaction
pair_style      mace
pair_coeff      * * {model_file} Li Ga

# Settings
neighbor        2.0 bin
neigh_modify    delay 10 every 1

# Define Groups for MSD
group           Li type {li_type_id}

# Minimization
minimize        0.0 1.0e-3 1000 1000

# Equilibration (NPT @ 300K - ambient condition for diffusion)
reset_timestep  0
timestep        0.002
velocity        all create 300.0 4928459 dist gaussian
fix             1 all npt temp 300.0 300.0 0.1 iso 1.0 1.0 1.0

# MSD Computation
# Calculate MSD for the 'Li' group
compute         msdLi Li msd
# Save MSD to file every 100 steps (averaged over 1 step to keep raw data)
fix             msd_out all ave/time 1 1 100 c_msdLi[4] file msd_li.dat title1 "TimeStep MSD_Li"

# Output
thermo          100
thermo_style    custom step temp pe ke etotal press vol c_msdLi[4]
dump            1 all custom 100 dump.lammpstrj id type x y z fx fy fz

run             500000 # 1 ns run for diffusion stats
""")

# --- MAIN LOOP ---
print(f"Processing structures from {input_dir}...")
os.makedirs(output_base, exist_ok=True)

for filename in os.listdir(input_dir):
    if not filename.endswith(".cif"): continue
    
    struct_name = os.path.splitext(filename)[0]
    file_path = os.path.join(input_dir, filename)
    
    try:
        struct = Structure.from_file(file_path)
        if not struct.is_ordered:
            struct = get_random_alloy(struct)
            
        current_atoms = len(struct)
        if current_atoms < min_atoms:
            scale_factor = (min_atoms / current_atoms) ** (1/3)
            s = math.ceil(scale_factor)
            struct.make_supercell([s, s, s])
            
        print(f"  {struct_name}: {len(struct)} atoms (Supercell {s}x{s}x{s})")
        
        task_dir = os.path.join(output_base, struct_name)
        os.makedirs(task_dir, exist_ok=True)
        
        # Create LammpsData to get atom type mapping
        ld = LammpsData.from_structure(struct,atom_style="atomic")
        ld.write_file(os.path.join(task_dir, "data.lammps"))
        
        # Extract Element -> Type ID mapping from pymatgen structure
        # Pymatgen sorts elements alphabetically usually.
        # We need to know which ID corresponds to Li for the 'group' command.
        # Get sorted list of elements in the structure
        elements = sorted(list(struct.composition.get_el_amt_dict().keys()))
        # Create map: {'Ga': 1, 'Li': 2} (Example)
        atom_map = {el: i+1 for i, el in enumerate(elements)}
        
        write_lammps_input(task_dir, struct_name, model_filename, atom_map)
        
    except Exception as e:
        print(f"  [Skipped] {filename}: {e}")

print("Done generating inputs with MSD tracking.")

Processing structures from LiGa/structures...
  Li2Ga7_mp-1222769: 6174 atoms (Supercell 7x7x7)
  LiGa3_mp-863257: 5324 atoms (Supercell 11x11x11)
  Li_mp-1018134: 5184 atoms (Supercell 12x12x12)
  Ga_mp-142: 5324 atoms (Supercell 11x11x11)
  LiGa_mp-1307: 5324 atoms (Supercell 11x11x11)
  Li2Ga_mp-29210: 6000 atoms (Supercell 10x10x10)
  Li3Ga14_mp-1222489: 5831 atoms (Supercell 7x7x7)
  Li5Ga4_mp-1205930: 6561 atoms (Supercell 9x9x9)
  Li3Ga2_mp-9568: 5000 atoms (Supercell 10x10x10)
Done generating inputs with MSD tracking.


In [18]:
import os
import glob
from dpdispatcher import Machine, Resources, Task, Submission

# --- CONFIG ---
work_base = "LiGa/MD_simulation/"
model_file = "/dssg/home/acct-umjbsh/umjbsh-sunzht/MACE_alloy/LiGa/MD_simulation/LiGa_MACE_FT.model-lammps.pt" # Must be in current dir!
job_name_prefix = "LiGa_MD"


# --- RESOURCES ---
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)
commands=[
        "ulimit -s unlimited &&", 
        "ulimit -l unlimited &&",
        "export OMP_NUM_THREADS=16 &&",
        "export MKL_NUM_THREADS=16 &&",
        "export OMP_PROC_BIND=spread && ",     # Important for hybrid
        "export OMP_PLACES=threads && ",
        "conda activate mace2 && ",
        "mpirun -np 16 lmp -i in.lammps"        
]

resources = Resources(
    number_node=4,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g",  # This sets the partition
    group_size=1,          # 1 task per job (no bundling)
    module_purge=True,
    module_list=["intel-oneapi-mpi/2021.4.0","intel-oneapi-compilers/2021.4.0","intel-mkl/2020.4.304","cuda/12.2.2"],
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=sunzhetao1997@sjtu.edu.cn",
        "#SBATCH --ntasks-per-node=64",
        "#SBATCH -N 4",
    ]
)

# --- COLLECT TASKS ---
task_list = []
print(f"Scanning {work_base}...")

# We need to resolve the absolute path of the model to symlink/copy it correctly
abs_model_path = os.path.abspath(model_file)

if not os.path.exists(abs_model_path):
    print(f"Error: Model file {abs_model_path} not found! Run 'mace_create_lammps_model' first.")
    exit()

for folder in glob.glob(os.path.join(work_base, "*")):
    if os.path.isdir(folder):
        # Create Task
        t = Task(
            command = ''.join(commands),
            task_work_path=os.path.relpath(folder) # Relative path preferred
            # forward_files=["in.lammps", "data.lammps"], 
            # backward_files=["dump.lammpstrj", "log.lammps"]
        )
        
#         # Manually add the model file to forward_files list 
#         # Since it lives in root but needs to go to task folder
#         # Dpdispatcher handles this if we pass the path
#         t.forward_files.append(abs_model_path)
        
        task_list.append(t)

print(f"Found {len(task_list)} tasks.")
for folder in glob.glob(os.path.join(work_base, "*")):
    print(folder)

Scanning LiGa/MD_simulation/...
Found 9 tasks.
LiGa/MD_simulation/LiGa3_mp-863257
LiGa/MD_simulation/Li2Ga_mp-29210
LiGa/MD_simulation/Li2Ga7_mp-1222769
LiGa/MD_simulation/LiGa_MACE_FT.model-lammps.pt
LiGa/MD_simulation/Ga_mp-142
LiGa/MD_simulation/Li3Ga14_mp-1222489
LiGa/MD_simulation/Li3Ga2_mp-9568
LiGa/MD_simulation/LiGa_mp-1307
LiGa/MD_simulation/Li5Ga4_mp-1205930
LiGa/MD_simulation/Li_mp-1018134


In [19]:
task_list[1:2]

[{'command': 'ulimit -s unlimited &&ulimit -l unlimited &&export OMP_NUM_THREADS=16 &&export MKL_NUM_THREADS=16 &&export OMP_PROC_BIND=spread && export OMP_PLACES=threads && conda activate mace2 && mpirun -np 16 lmp -i in.lammps', 'task_work_path': 'LiGa/MD_simulation/Li2Ga_mp-29210', 'forward_files': [], 'backward_files': [], 'outlog': 'log', 'errlog': 'err'}]

In [20]:
# --- SUBMIT ---

if task_list:
    submission = Submission(
        work_base=os.getcwd(), # Run from root
        machine=machine,
        resources=resources,
        task_list=task_list[1:2]
    )

    print("Submitting MD jobs...")
    submission.run_submission(exit_on_submit=True)
    print("Submission finished.")
else:
    print("No tasks found.")

Submitting MD jobs...
2025-11-21 23:07:58,334 - INFO : info:check_all_finished: False
2025-11-21 23:07:58,361 - INFO : job: 319e09db41180cffcde861ec4f39fc4c133f8f26 submit; job_id is 50281898
2025-11-21 23:07:59,385 - INFO : submission succeeded: 0262d4258edbddb5607cca2047345637a3db5e9e
2025-11-21 23:07:59,387 - INFO : at /dssg/home/acct-umjbsh/umjbsh-sunzht/MACE_alloy
Submission finished.


In [24]:
commands=[
        "ulimit -s unlimited &&", 
        "ulimit -l unlimited &&",
        "export OMP_NUM_THREADS=32 &&",
        "export MKL_NUM_THREADS=32 &&",
        "export OMP_PROC_BIND=spread && ",     # Important for hybrid
        "export OMP_PLACES=threads && ",
        "conda activate mace2 && ",
        "mpirun -np 8 lmp -i in.lammps"        
]
''.join(commands)

'ulimit -s unlimited &&ulimit -l unlimited &&export OMP_NUM_THREADS=32 &&export MKL_NUM_THREADS=32 &&export OMP_PROC_BIND=spread && export OMP_PLACES=threads && conda activate mace2 && mpirun -np 4 lmp -i in.lammps'

### Guoyong think the low diffusivity is caused by the Disproportionation reaction ${Li_3Ga_2 \to LiGa+Li_2Ga}$

In [ ]:
import os
import math
import warnings
import numpy as np
from pymatgen.core import Structure
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.io.lammps.data import LammpsData

warnings.filterwarnings("ignore")

# --- CONFIGURATION ---
structures_dir = "LiGa/structures"
output_dir = "LiGa/Interface_MD"
min_atoms = 3000  # Target size

# UPDATED SCAN LIST
# Includes (1,1,1) for FCC-like LiGa and various low-index planes
SCAN_MILLERS = [
    (1, 1, 1),  # Densest plane for LiGa (Cubic)
    (1, 1, 0),
    (1, 0, 0),
    (0, 0, 1),
    (2, 1, 0),  # Search slightly higher indices for Li2Ga match
    (2, 1, 1)
]

def find_structure_by_formula(folder, formula):
    """ Finds the first CIF file in the folder matching the formula. """
    for fname in os.listdir(folder):
        if fname.endswith(".cif") and formula in fname:
            return os.path.join(folder, fname)
    return None

def create_interface():
    print("Searching for LiGa and Li2Ga structures...")
    file_LiGa = find_structure_by_formula(structures_dir, "LiGa")
    file_Li2Ga = find_structure_by_formula(structures_dir, "Li2Ga")
    
    if not file_LiGa or not file_Li2Ga:
        print(f"Error: Could not find LiGa or Li2Ga files in {structures_dir}")
        return None

    print(f"Found: {os.path.basename(file_LiGa)} & {os.path.basename(file_Li2Ga)}")
    
    s1 = Structure.from_file(file_LiGa).to_conventional()
    s2 = Structure.from_file(file_Li2Ga).to_conventional()
    
    # Relaxed Tolerances for Strain Matching
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.25, # Allow 25% area mismatch (flexible)
        max_area=2000,
        max_length_tol=0.15,     # 15% length mismatch
        max_angle_tol=0.15       # 15% angle mismatch
    )
    
    best_interface = None
    min_strain = float('inf')
    best_orientation = None

    print(f"\nScanning Miller Index Space: {len(SCAN_MILLERS)} planes")
    print("-" * 60)

    # --- LOOP OVER ORIENTATIONS ---
    # We check every combination of substrate/film orientation
    for hkl_sub in SCAN_MILLERS:
        for hkl_film in SCAN_MILLERS:
            
            # Skip redundant checks if simple cubic symmetry implies they are same
            # (But safe to check for general structures)
            
            try:
                builder = CoherentInterfaceBuilder(
                    substrate_structure=s1,
                    film_structure=s2,
                    film_miller=hkl_film, 
                    substrate_miller=hkl_sub,
                    zslgen=zsl
                )
                
                # Get interfaces (no termination constraint)
                interfaces = list(builder.get_interfaces())
                
                if not interfaces:
                    continue
                
                # Find lowest strain in this batch
                interfaces.sort(key=lambda x: x.separation)
                current_best = interfaces[0]
                strain = current_best.separation
                
                print(f"Match: LiGa {hkl_sub} // Li2Ga {hkl_film} -> Strain: {strain:.4f} Å")
                
                if strain < min_strain:
                    min_strain = strain
                    best_interface = current_best
                    best_orientation = f"LiGa {hkl_sub} // Li2Ga {hkl_film}"
                    
            except Exception:
                continue

    print("-" * 60)
    
    if best_interface:
        print(f"BEST INTERFACE: {best_orientation}")
        print(f"Strain: {min_strain:.4f} Å")
        
        struct = best_interface.copy()
        
        # Scale to Target Size
        current_atoms = len(struct)
        if current_atoms < min_atoms:
            scale_factor = (min_atoms / current_atoms) ** (1/3)
            s = math.ceil(scale_factor)
            # Elongate Z to ensure bulk-like regions away from interface
            struct.make_supercell([s, s, s*2]) 
        
        print(f"Final Supercell: {len(struct)} atoms")
        return struct
    else:
        print("Failed to find any coherent interface. Try increasing 'max_area' in ZSLGenerator.")
        return None

# --- MAIN EXECUTION ---
os.makedirs(output_dir, exist_ok=True)
interface_struct = create_interface()

if interface_struct:
    # Write Data
    ld = LammpsData.from_structure(interface_struct, atom_style="atomic")
    ld.write_file(os.path.join(output_dir, "data.lammps"))
    
    print(f"\nSuccess! Interface data written to {os.path.join(output_dir, 'data.lammps')}")
    
    elements = sorted(list(interface_struct.composition.get_el_amt_dict().keys()))
    print("Atom Type Mapping:")
    for i, el in enumerate(elements):
        print(f"  Type {i+1} = {el}")
else:
    print("Generation Failed.")

Searching for LiGa and Li2Ga structures...
Found: LiGa3_mp-863257.cif & Li2Ga7_mp-1222769.cif

Scanning Miller Index Space: 6 planes
------------------------------------------------------------
